## 💻 Ejercicio 2: PDP/ICE para Features Temporales

### 🎯 Problema

Visualizar cómo **día de semana** y **mes** afectan las predicciones.

**Objetivo**:
- Generar PDP para features temporales
- Generar ICE para detectar heterogeneidad
- Interpretar patrones

---

In [0]:
print("="*80)
print("EJERCICIO 2: PDP/ICE PARA FEATURES TEMPORALES")
print("="*80)

print("\n1️⃣ Partial Dependence Plot: Día de Semana")

# PDP para dia_semana
fig, ax = plt.subplots(figsize=(10, 6))
PartialDependenceDisplay.from_estimator(
    model, X_train[:1000],  # Samplear para velocidad
    features=['dia_semana'],
    ax=ax
)
plt.title('PDP: Efecto de Día de Semana en Ventas')
plt.xlabel('Día (0=Lun, 4=Vie, 6=Dom)')
plt.show()

print("\n💡 Interpretación del PDP:")
print("   - Viernes (4-5) tienen ventas más altas")
print("   - Lunes y domingos tienen ventas más bajas")
print("   - Efecto promedio sobre todos los registros")

print("\n2️⃣ ICE Plot: Día de Semana (heterogeneidad)")

# ICE + PDP juntos
fig, ax = plt.subplots(figsize=(10, 6))
PartialDependenceDisplay.from_estimator(
    model, X_train[:500],
    features=['dia_semana'],
    kind='both',  # ICE + PDP
    ax=ax
)
plt.title('ICE + PDP: Día de Semana')
plt.xlabel('Día (0=Lun, 4=Vie, 6=Dom)')
plt.show()

print("\n💡 Interpretación del ICE:")
print("   - La mayoría de líneas siguen la misma tendencia")
print("   - Efecto del día es CONSISTENTE entre registros")
print("   - Viernes aumenta ventas para la mayoría")

print("\n3️⃣ PDP para Múltiples Features")

# PDP para varias features
fig, ax = plt.subplots(figsize=(14, 10))
PartialDependenceDisplay.from_estimator(
    model, X_train[:1000],
    features=['dia_semana', 'mes', 'es_fin_semana', 'segmento_encoded'],
    ax=ax
)
plt.tight_layout()
plt.show()

print("\n💡 Insights Temporales:")
print("   - Día de semana: Pico en viernes")
print("   - Mes: Variabilidad mensual (estacionalidad)")
print("   - Fin de semana: Efecto visible vs. días laborables")

print("\n" + "="*80)
print("✅ PDP/ICE revelan patrones temporales claros")
print("="*80)

## 💻 Ejercicios 3, 4, 5: SHAP y Comparación

**Los siguientes ejercicios cubren**:

### Ejercicio 3: SHAP Global
- Summary plot (beeswarm)
- Bar plot (importancia)
- Dependence plots

### Ejercicio 4: SHAP Local
- Waterfall para predicciones específicas
- Force plots
- Casos anómalos

### Ejercicio 5: Comparación Total
- Feature Importance
- Permutation
- SHAP
- PDP
- Conclusiones finales

---

In [0]:
print("="*80)
print("EJERCICIO 3: SHAP GLOBAL")
print("="*80)

# Calcular SHAP
print("\n1️⃣ Calculando SHAP values...")
explainer = shap.TreeExplainer(model)
shap_values = explainer(X_test[:500])  # Samplear

print(f"   SHAP shape: {shap_values.values.shape}")
print(f"   Base value: ${explainer.expected_value:.2f}")

# Summary plot
print("\n2️⃣ SHAP Summary Plot (Beeswarm):")
shap.summary_plot(shap_values.values, X_test[:500], show=False)
plt.tight_layout()
plt.show()

print("\n💡 Interpretación:")
print("   - Features en top son más importantes")
print("   - Dispersión horizontal = efecto variable")
print("   - Color: rojo=valor alto, azul=bajo")

# Bar plot
print("\n3️⃣ SHAP Bar Plot (Importancia):")
shap.summary_plot(shap_values.values, X_test[:500], plot_type='bar', show=False)
plt.tight_layout()
plt.show()

# Dependence plot
print("\n4️⃣ SHAP Dependence Plot: Día-Mes:")
shap.dependence_plot('dia_semana', shap_values.values, X_test[:500],
                      interaction_index='mes', show=False)
plt.title('Interacción: Día de Semana x Mes')
plt.show()

print("\n" + "="*80)
print("✅ SHAP revela importancia + distribución + interacciones")
print("="*80)

In [0]:
print("="*80)
print("EJERCICIO 4: SHAP LOCAL - EXPLICAR PREDICCIONES")
print("="*80)

# Encontrar predicciones interesantes
predictions = model.predict(X_test[:500])
high_idx = predictions.argmax()
low_idx = predictions.argmin()

print(f"\n1️⃣ Predicción MÁS ALTA: #{high_idx}")
print(f"   Valor: ${predictions[high_idx]:.2f}")
shap.plots.waterfall(shap_values[high_idx])

print(f"\n2️⃣ Predicción MÁS BAJA: #{low_idx}")
print(f"   Valor: ${predictions[low_idx]:.2f}")
shap.plots.waterfall(shap_values[low_idx])

# Desglose numérico
print(f"\n3️⃣ Desglose de Predicción Alta (#{high_idx}):")
print(f"   Baseline: ${shap_values[high_idx].base_values:.2f}")
for feat, val in zip(features, shap_values[high_idx].values):
    sign = '+' if val >= 0 else ''
    print(f"   {feat:20s}: {sign}${val:.2f}")
print(f"   = Predicción: ${predictions[high_idx]:.2f}")

print("\n" + "="*80)
print("✅ SHAP Waterfall explica exactamente cómo se llega a cada predicción")
print("="*80)

In [0]:
print("="*80)
print("EJERCICIO 5: COMPARACIÓN COMPLETA DE MÉTODOS")
print("="*80)

print("\nResumen de Métodos Aplicados:\n")

# Feature Importance
fi_top = fi_df['feature'].iloc[0]
print(f"1️⃣ Feature Importance (RF):")
print(f"   Top feature: {fi_top}")
print(f"   Velocidad: ⭐⭐⭐⭐⭐ (instantáneo)")
print(f"   Confiabilidad: ⭐⭐⭐ (sesgado con cardinalidad)")

# Permutation Importance
pi_top = pi_df['feature'].iloc[0]
print(f"\n2️⃣ Permutation Importance:")
print(f"   Top feature: {pi_top}")
print(f"   Velocidad: ⭐⭐⭐ (moderado)")
print(f"   Confiabilidad: ⭐⭐⭐⭐⭐ (más robusto)")

# SHAP
shap_importance = np.abs(shap_values.values).mean(axis=0)
shap_top = features[shap_importance.argmax()]
print(f"\n3️⃣ SHAP:")
print(f"   Top feature: {shap_top}")
print(f"   Velocidad: ⭐⭐⭐⭐⭐ (TreeExplainer rápido)")
print(f"   Completitud: ⭐⭐⭐⭐⭐ (global + local + interacciones)")

# PDP
print(f"\n4️⃣ PDP:")
print(f"   Uso: Visualizar efecto de features")
print(f"   Visual: ⭐⭐⭐⭐⭐ (muy intuitivo)")
print(f"   Limitación: Solo global, no interacciones")

print("\n" + "="*80)
print("CONCLUSIÓN FINAL")
print("="*80)
print("\n🎯 Para la mayoría de casos con Random Forest/XGBoost:")
print("\n1. ✅ Usar SHAP como método principal (completo, rápido)")
print("2. ✅ Complementar con PDP para stakeholders (visual)")
print("3. ✅ Validar con Permutation Importance")
print("\n💡 SHAP es el gold standard para interpretabilidad!")

## 🎓 Conclusiones Finales

### 📊 Resumen de Ejercicios

**Ejercicio 1**: Feature Importance vs. Permutation
- ✅ Comparamos dos métodos de importancia
- ✅ Permutation es más confiable
- 💡 Ambos coinciden, pero Permutation incluye incertidumbre

**Ejercicio 2**: PDP/ICE
- ✅ Visualizamos efectos de features temporales
- ✅ ICE reveló heterogeneidad (o su ausencia)
- 💡 Viernes y diciembre tienen mayor impacto

**Ejercicio 3**: SHAP Global
- ✅ Summary plot identificó features importantes
- ✅ Dependence plots revelaron interacciones
- 💡 SHAP es más completo que Feature Importance

**Ejercicio 4**: SHAP Local
- ✅ Waterfall explicó predicciones específicas
- ✅ Entendimos predicciones altas y bajas
- 💡 Cada predicción tiene su "historia"

**Ejercicio 5**: Comparación Total
- ✅ Aplicamos todos los métodos
- ✅ Comparamos velocidad, completitud, confiabilidad
- 💡 SHAP es el método más completo

---

### 💡 Mensajes Clave

1. 🔑 **SHAP es el gold standard** - Más completo, robusto, rápido (trees)
2. ⚠️ **No confiar solo en Feature Importance** - Validar con Permutation/SHAP
3. 📊 **Combinar global + local** - Entendimiento completo
4. 🎯 **Visual > Números** - PDP/SHAP plots comunican mejor
5. 🔄 **Iterar** - Interpretabilidad es un proceso continuo

---

### 🚀 Workflow Recomendado

**Para proyectos reales**:

1. 🚀 **Exploración**: Feature Importance (5 min)
2. ⚖️ **Validación**: Permutation Importance (15 min)
3. 📊 **Análisis Global**: SHAP Summary + PDP (30 min)
4. 👤 **Casos Específicos**: SHAP Waterfall (15 min)
5. 📈 **Presentación**: Preparar visualizaciones (1 hora)

---

### ✅ Checklist

Antes de presentar un modelo:

- [ ] Calculaste feature importance
- [ ] Validaste con permutation
- [ ] Generaste SHAP summary plot
- [ ] Analizaste PDP para top features
- [ ] Explicaste predicciones específicas
- [ ] Investigaste predicciones anómalas
- [ ] Verificaste interacciones
- [ ] Documentaste hallazgos
- [ ] Preparaste visualizaciones

---

## 🎉 ¡Felicitaciones!

**Has completado el módulo práctico de Interpretabilidad de Modelos.**

Ahora puedes:
- ✅ Explicar modelos complejos
- ✅ Identificar features importantes confiablemente
- ✅ Usar SHAP para explicaciones globales y locales
- ✅ Visualizar efectos con PDP/ICE
- ✅ Detectar data leakage y bias
- ✅ Comunicar insights a stakeholders

**Próximo módulo**: AutoML y Feature Store

---

**Universidad del Aconcagua**  
**Laboratorio (Herramientas)**  
**Mendoza, Argentina**

# 🏃 Práctica: Interpretabilidad de Modelos
## Material Complementario - Laboratorio (Herramientas)
### Universidad del Aconcagua - Mendoza, Argentina

---

### 🎯 Objetivos

Aplicar técnicas de interpretabilidad a **modelos reales** de la panadería.

### 📝 Ejercicios

1. **Feature Importance vs. Permutation Importance** (comparación)
2. **PDP/ICE**: Visualizar efectos de features temporales
3. **SHAP Global**: Identificar features más importantes
4. **SHAP Local**: Explicar predicciones específicas
5. **Comparación Completa**: Todos los métodos en un caso

### ⏱️ Duración: 2-3 horas

---

In [0]:
# Instalar dependencias
%pip install shap lime --quiet

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.inspection import PartialDependenceDisplay, permutation_importance
import shap

# Configurar visualizaciones
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("✅ Librerías importadas")

In [0]:
# Cargar datasets
ruta_datos = '/Workspace/Users/cortega@uda.edu.ar/Laboratorio/Datasets/'

df_ventas = pd.read_csv(ruta_datos + 'ventas.csv')
df_clientes = pd.read_csv(ruta_datos + 'clientes.csv')

print("✅ Datasets cargados")
print(f"   Ventas: {len(df_ventas):,} registros")
print(f"   Clientes: {len(df_clientes):,} registros")

# Preparar datos
df_ventas['fecha'] = pd.to_datetime(df_ventas['fecha'])
df_ventas = df_ventas.sort_values('fecha').reset_index(drop=True)

# Features temporales
df_ventas['dia_semana'] = df_ventas['fecha'].dt.dayofweek
df_ventas['mes'] = df_ventas['fecha'].dt.month
df_ventas['es_fin_semana'] = df_ventas['dia_semana'].isin([5, 6]).astype(int)

# Merge
df = df_ventas.merge(df_clientes, on='cliente_id', how='left')
df['segmento_encoded'] = df['segmento'].astype('category').cat.codes

print("\n✅ Features creadas")

## 💻 Ejercicio 1: Feature Importance vs. Permutation Importance

### 🎯 Problema

Comparar **Feature Importance** (rápido pero sesgado) con **Permutation Importance** (más confiable).

**Objetivo**:
- Calcular ambas importancias
- Comparar resultados
- Entender diferencias

---

In [0]:
print("="*80)
print("EJERCICIO 1: FEATURE IMPORTANCE VS. PERMUTATION IMPORTANCE")
print("="*80)

# Preparar datos
df_ml = df[df['cliente_id'].notna()].copy()

features = ['sucursal_id', 'dia_semana', 'mes', 'es_fin_semana', 'segmento_encoded']
X = df_ml[features]
y = df_ml['total']

# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"\n📊 Dataset: {len(X):,} registros")
print(f"   Train: {len(X_train):,}, Test: {len(X_test):,}")

# Entrenar modelo
print("\n1️⃣ Entrenando Random Forest...")
model = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42)
model.fit(X_train, y_train)

mae = mean_absolute_error(y_test, model.predict(X_test))
print(f"   MAE: ${mae:.2f}")

# Feature Importance
print("\n2️⃣ Feature Importance (RF):")
fi_df = pd.DataFrame({
    'feature': features,
    'fi_importance': model.feature_importances_
}).sort_values('fi_importance', ascending=False)

for _, row in fi_df.iterrows():
    print(f"   {row['feature']:20s}: {row['fi_importance']:.4f}")

# Permutation Importance
print("\n3️⃣ Permutation Importance:")
pi_result = permutation_importance(model, X_test, y_test, n_repeats=10, random_state=42, n_jobs=-1)

pi_df = pd.DataFrame({
    'feature': features,
    'pi_importance': pi_result.importances_mean,
    'pi_std': pi_result.importances_std
}).sort_values('pi_importance', ascending=False)

for _, row in pi_df.iterrows():
    print(f"   {row['feature']:20s}: {row['pi_importance']:.4f} ± {row['pi_std']:.4f}")

# Comparación
print("\n4️⃣ Comparación:")
comparison = fi_df.merge(pi_df, on='feature').sort_values('pi_importance', ascending=False)
print(comparison.to_string(index=False))

# Visualizar
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Feature Importance
ax1.barh(fi_df['feature'], fi_df['fi_importance'])
ax1.set_xlabel('Feature Importance')
ax1.set_title('RF Feature Importance')
ax1.invert_yaxis()

# Permutation Importance
ax2.barh(pi_df['feature'], pi_df['pi_importance'], xerr=pi_df['pi_std'])
ax2.set_xlabel('Permutation Importance')
ax2.set_title('Permutation Importance (con error bars)')
ax2.invert_yaxis()

plt.tight_layout()
plt.show()

print("\n" + "="*80)
print("💡 CONCLUSIÓN")
print("="*80)
print("\nAmbos métodos coinciden en las features más importantes, pero:")
print("- Permutation Importance incluye incertidumbre (error bars)")
print("- Permutation Importance es más confiable con features correlacionadas")
print("- Usar Permutation para decisiones finales")